# DataForge Arena - GRPO Training on Colab

**Meta PyTorch + HuggingFace OpenEnv Hackathon 2026**

This notebook trains an LLM to repair corrupted enterprise data using Group Relative Policy Optimization (GRPO).

Requirements: GPU runtime (T4 minimum). Go to **Runtime -> Change runtime type -> T4 GPU**.

## 1. Setup & Install

In [ ]:
# Clone repo, or refresh it if the runtime was restarted and /content/dataforge-arena already exists.
import os
import subprocess

REPO_DIR = '/content/dataforge-arena'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/vivekyarra/dataforge-arena.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')


In [ ]:
# Install a pinned, Colab-safe GRPO training stack
# If this cell restarts the runtime, reconnect and run the notebook from the top once more.
import os
import sys

if 'torch' in sys.modules:
    print('Torch is already loaded in this runtime. Restarting once so pinned CUDA packages load cleanly.')
    os.kill(os.getpid(), 9)

# Unsloth 2026.4.8 requires torch < 2.11 and TRL <= 0.24, so keep this stack together.
!python -m pip install -q --upgrade --extra-index-url https://download.pytorch.org/whl/cu128 torch==2.10.0+cu128 torchvision==0.25.0+cu128 torchaudio==2.10.0+cu128

# Install the matching RL stack plus TRL optional imports that can be imported by trainer callbacks.
# Keep pydantic >=2.12 for Colab's preinstalled mcp/google-adk, and avoid click 8.2.* because rasterio rejects it.
# torchao 0.16.0 is the torch 2.10-compatible release and avoids noisy cpp-extension warnings.
!python -m pip install -q --upgrade --extra-index-url https://download.pytorch.org/whl/cu128 unsloth==2026.4.8 trl==0.24.0 torchao==0.16.0 transformers datasets accelerate bitsandbytes openenv faker mergekit llm-blender==0.0.2 weave==0.52.37 pydantic==2.12.5 click==8.1.8

# Fail fast here instead of 20 minutes later in training.
import importlib
from importlib.metadata import version
import torch
import transformers.utils.hub

if not hasattr(transformers.utils.hub, 'TRANSFORMERS_CACHE'):
    transformers.utils.hub.TRANSFORMERS_CACHE = os.getenv('HF_HOME', '/tmp/hf_cache')

for module_name in ('llm_blender', 'weave', 'torchao'):
    importlib.import_module(module_name)

from trl import GRPOConfig, GRPOTrainer
from unsloth import FastLanguageModel

major, minor = map(int, torch.__version__.split('+')[0].split('.')[:2])
assert (major, minor) == (2, 10), f'torch 2.10.x required for this Unsloth stack, found {torch.__version__}'
assert tuple(map(int, version('pydantic').split('.')[:2])) >= (2, 12), f'pydantic >=2.12 required, found {version("pydantic")}'
assert not version('click').startswith('8.2.'), f'click 8.2.* conflicts with Colab rasterio, found {version("click")}'
print(f'Dependencies ready: torch={torch.__version__}, cuda={torch.version.cuda}, pydantic={version("pydantic")}, click={version("click")}')


In [ ]:
# Generate clean training data
!python training/generate_data.py

## 2. Verify GPU & Model Selection

In [ ]:
!nvidia-smi
print('---')
!python -c "from training.model_config import detect_gpu, select_model; g=detect_gpu(); m=select_model(g); print(f'GPU: {g}'); print(f'Model: {m[\"label\"]}')"

## 3. Run Tests (must all pass)

In [ ]:
!python -m pytest -q

## 4. Quick Sanity Check - Single Episode

In [ ]:
import pandas as pd
from environment.env import DataForgeEnv, SurgeonAction
from environment.corruptor import Corruptor
from environment.reward import RewardComputer
from environment.schemas import HEALTHCARE_SCHEMA

df = pd.read_csv('data/healthcare_clean.csv')
env = DataForgeEnv(Corruptor(), HEALTHCARE_SCHEMA, df)
obs = env.reset()
action = SurgeonAction(reasoning='test null', tool_id=0, column=2, row_id=0)
obs2, reward, done, info = env.step(action)
print(f'Reward: {reward}')
print(f'Done: {done}')

clean_sample = df.head(5).copy()
dirty_sample = clean_sample.copy()
dirty_sample.at[0, 'age'] = pd.NA
age_col = list(clean_sample.columns).index('age')
efficiency_action = SurgeonAction(
    reasoning='age is null because the value is missing',
    tool_id=0,
    column=age_col,
    row_id=0,
)
efficiency = RewardComputer()._score_efficiency(
    efficiency_action,
    clean_sample,
    clean_sample,
    previous_state=dirty_sample,
)
assert efficiency == 0.5, f'Efficiency repair-target bonus missing: {efficiency}'
print('Episode loop and efficiency reward smoke tests work!')

## 5. GRPO Training

This runs the full training loop. On a T4: ~60 min for 80 steps. The install cell above pins the Unsloth-compatible torch, TRL, and optional TRL imports so `GRPOTrainer` imports cleanly before training starts.


In [ ]:
# Suppress noisy warnings
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Run training
!python training/train_grpo.py

## 6. Evaluation - Before/After Numbers

In [ ]:
!python eval/evaluate.py --episodes 20 --tier 1
print('---')
!python eval/evaluate.py --episodes 10 --tier 2
print('---')
!python eval/evaluate.py --episodes 10 --tier 3

## 7. Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

try:
    log = pd.read_csv('logs/training_log.csv')
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.patch.set_facecolor('#0a0a0f')
    
    for ax in axes:
        ax.set_facecolor('#111118')
        ax.tick_params(colors='#94a3b8')
        ax.spines['bottom'].set_color('#1e293b')
        ax.spines['left'].set_color('#1e293b')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    axes[0].plot(log['step'], log['total_reward'], color='#10b981', linewidth=2)
    axes[0].set_title('Total Reward', color='#e2e8f0', fontsize=14)
    axes[0].set_xlabel('Step', color='#94a3b8')
    axes[0].set_ylabel('Reward', color='#94a3b8')
    axes[0].axhline(y=0, color='#374151', linestyle='--', alpha=0.5)
    
    axes[1].plot(log['step'], log['accuracy_delta'], color='#38bdf8', linewidth=2)
    axes[1].set_title('Accuracy Delta', color='#e2e8f0', fontsize=14)
    axes[1].set_xlabel('Step', color='#94a3b8')
    axes[1].set_ylabel('Delta Reward Component', color='#94a3b8')
    axes[1].axhline(y=0, color='#374151', linestyle='--', alpha=0.5)
    
    axes[2].plot(log['step'], log['efficiency'], color='#f59e0b', linewidth=2)
    axes[2].set_title('Efficiency Targeting', color='#e2e8f0', fontsize=14)
    axes[2].set_xlabel('Step', color='#94a3b8')
    axes[2].set_ylabel('Reward Component', color='#94a3b8')
    axes[2].axhline(y=0, color='#374151', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0f')
    plt.show()
    print(f"Accuracy delta first -> final: {log['accuracy_delta'].iloc[0]:+.4f} -> {log['accuracy_delta'].iloc[-1]:+.4f}")
    print(f"Efficiency first -> final: {log['efficiency'].iloc[0]:+.4f} -> {log['efficiency'].iloc[-1]:+.4f}")
    print('Saved to training_curves.png')
except FileNotFoundError:
    print('No training log found. Run training first.')

## 8. Save & Download

In [ ]:
# Zip the trained model for download
!zip -r /content/dataforge-trained.zip outputs/dataforge-surgeon/ logs/ eval/results.json training_curves.png 2>/dev/null || true
print('Download: /content/dataforge-trained.zip')

# Show final stats
import json
try:
    with open('eval/results.json') as f:
        r = json.load(f)
    print(f"\nSurgeon advantage accuracy delta: {r['surgeon_advantage_accuracy_delta']*100:+.2f}% over random baseline")
except: pass